# LOS/NLOS labels da coordinate CSV/JSON (+ DEM)

Flusso Colab Genova (esegui le celle **in ordine**):

1. Setup GPU + clone + build `_bvh`
2. Dati (`points.csv`, NAV, mesh OSM + DEM)
3. Label LOS/NLOS
4. Viewer Cesium (raggi)
5. Skyline / skyplot + export mask
6. Download risultati

Script: `experiments/label_los_nlos_coords.py`

### Accortezze Genova
1. `--rx-alt-mode dem` (niente alt fissa 1.5 m ellissoidale)
2. DEM horizon (`--dem-path` / auto-download)
3. **`--geoid-correction egm96`** su labeling **e** mesh OSM (AWS DEM ortometrico)
4. Stesso DEM in `fetch_osm_buildings_bbox.py`
5. Bbox storica città: `44.385 / 8.880 / 44.445 / 8.980`

**Runtime:** GPU (T4) · branch `feature/cuda-preprocess-area-map`  
**Repo:** `https://github.com/andrea444460/gnss_gpu_los_nlos`


In [ ]:
# 1) GPU
!nvidia-smi


In [ ]:
# 2) Clone / aggiorna repo
import os, shutil, subprocess

REPO = "/content/gnss_gpu"
BRANCH = "feature/cuda-preprocess-area-map"
REPO_URL = "https://github.com/andrea444460/gnss_gpu_los_nlos.git"
SCRIPT = f"{REPO}/experiments/label_los_nlos_coords.py"

def _has_script():
    return os.path.isfile(SCRIPT)

if os.path.isdir(REPO) and not _has_script():
    print("Clone incompleto → rimuovo")
    shutil.rmtree(REPO)

if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    subprocess.run(["git", "-C", REPO, "fetch", REPO_URL, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", REPO_URL, BRANCH], check=True)

os.chdir(REPO)
assert _has_script(), f"Manca {SCRIPT}"
print("OK:", SCRIPT)
!git -C {REPO} rev-parse --short HEAD


In [ ]:
# 3) Dipendenze + build CUDA `_bvh`
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path("/content/gnss_gpu")
os.chdir(REPO)

!pip install -q numpy matplotlib pandas requests scipy rasterio pyproj pybind11
!apt-get -qq install -y cmake

arch = "75"
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        text=True,
    ).strip().splitlines()[0]
    major, minor = out.split(".")
    arch = f"{major}{minor}"
except Exception as exc:
    print("arch fallback sm_75:", exc)
print("CUDA arch:", arch)

pybind_dir = subprocess.check_output(
    [sys.executable, "-c", "import pybind11; print(pybind11.get_cmake_dir())"],
    text=True,
).strip()

build = REPO / "build"
if (build / "CMakeCache.txt").exists():
    cache = (build / "CMakeCache.txt").read_text(errors="ignore")
    bad = ("pybind11_DIR:PATH=" not in cache) or ("NOTFOUND" in cache and "pybind11" in cache.lower())
    if bad:
        print("CMake cache senza pybind11 → rebuild pulito")
        shutil.rmtree(build)
build.mkdir(exist_ok=True)
os.chdir(build)
!cmake .. -DCMAKE_CUDA_ARCHITECTURES={arch} -Dpybind11_DIR={pybind_dir}
!make -j$(nproc) _bvh
os.chdir(REPO)

pkg = REPO / "python" / "gnss_gpu"
sos = list(build.glob("_bvh*.so")) + list(build.glob("_bvh*.pyd"))
if not sos:
    sos = list(build.rglob("_bvh*.so")) + list(build.rglob("_bvh*.pyd"))
for src in sos:
    shutil.copy2(src, pkg / src.name)
    print("copied →", pkg / src.name)
if not sos:
    raise RuntimeError("Nessun _bvh*.so — controlla cmake/pybind11")


In [ ]:
# 4) Test import BVH
import os, sys, shutil
from pathlib import Path
import importlib

REPO = Path("/content/gnss_gpu")
os.chdir(REPO)
pkg = REPO / "python" / "gnss_gpu"
for src in (REPO / "build").rglob("_bvh*.so"):
    dst = pkg / src.name
    if not dst.exists() or dst.stat().st_mtime < src.stat().st_mtime:
        shutil.copy2(src, dst)

os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
sys.path = [str(REPO / "python")] + [
    p for p in sys.path if p not in (str(REPO / "python"), str(REPO / "build"))
]

import gnss_gpu
importlib.reload(gnss_gpu)
from gnss_gpu import _bvh
from gnss_gpu.bvh import BVHAccelerator
print("_bvh OK:", _bvh)
print("BVHAccelerator OK")


## 5) Dati

- `/content/los_work/data/` → `points.csv`, NAV `*MN.rnx`
- `/content/los_work/results/` → mesh, DEM, CSV, HTML, skyline

CSV: `id,lat,lon` (con `--rx-alt-mode dem` l’alt CSV è ignorata).


In [ ]:
# 5a) Cartelle + sample Genova (n5)
from pathlib import Path
import shutil

WORK = Path("/content/los_work")
DATA = WORK / "data"
OUT = WORK / "results"
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

example = DATA / "points.csv"
repo_sample = Path("/content/gnss_gpu/experiments/data/genova/2026-10-09_points_genua_n5.csv")
if not example.exists():
    if repo_sample.exists():
        shutil.copy2(repo_sample, example)
        print("Copiato sample Genua n5:", example)
    else:
        example.write_text(
            "id,lat,lon\n"
            "centro,44.4070,8.9330\n"
            "porto,44.4055,8.9280\n",
            encoding="utf-8",
        )
        print("Creato esempio:", example)
else:
    print("Esiste già:", example)
print("data/:", sorted(p.name for p in DATA.iterdir()))


In [ ]:
# 5b) Upload opzionale (points / NAV / triangles)
from google.colab import files
from pathlib import Path
import shutil

DATA = Path("/content/los_work/data")
DATA.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    dest = DATA / name
    shutil.move(name, dest)
    print("→", dest)


In [ ]:
# 5c) Download BRDC NAV (BKG) → data/
import gzip, shutil, urllib.request, urllib.error
from pathlib import Path
from datetime import datetime, timezone

DATA = Path("/content/los_work/data")
DATA.mkdir(parents=True, exist_ok=True)

YEAR, MONTH, DAY = 2026, 10, 9  # allinea a points / UTC labeling
dt = datetime(YEAR, MONTH, DAY, tzinfo=timezone.utc)
DOY = dt.timetuple().tm_yday
PRODUCTS = ("BRDC00WRD_R", "BRDC00IGS_R", "BRD400DLR_S")

def brdc_name(product, year, doy):
    return f"{product}_{year}{doy:03d}0000_01D_MN.rnx"

def urls(product, year, doy):
    name = brdc_name(product, year, doy) + ".gz"
    doy3 = f"{doy:03d}"
    return [
        f"https://igs.bkg.bund.de/root_ftp/IGS/BRDC/{year}/{doy3}/{name}",
        f"https://igs.bkg.bund.de/root_ftp/EUREF/BRDC/{year}/{doy3}/{name}",
    ]

nav_out = None
for product in PRODUCTS:
    rnx = DATA / brdc_name(product, YEAR, DOY)
    if rnx.exists() and rnx.stat().st_size > 0:
        print("reuse", rnx)
        nav_out = rnx
        break
    for url in urls(product, YEAR, DOY):
        print("try", url)
        try:
            with urllib.request.urlopen(url, timeout=120) as resp:
                data = resp.read()
        except urllib.error.HTTPError as e:
            print(" ", e.code)
            continue
        if data[:2] != b"\x1f\x8b":
            print("  not gzip")
            continue
        gz = DATA / (rnx.name + ".gz")
        gz.write_bytes(data)
        with gzip.open(gz, "rb") as fi, open(rnx, "wb") as fo:
            shutil.copyfileobj(fi, fo)
        print("wrote", rnx, f"({rnx.stat().st_size // 1024} KB)")
        nav_out = rnx
        break
    if nav_out is not None:
        break

assert nav_out is not None, f"BRDC non trovato per {YEAR} DOY {DOY:03d}"
print("NAV:", nav_out)


In [ ]:
# 5d) DEM + mesh OSM (stesso DEM + egm96)
import os, csv
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
os.chdir(REPO)

coords = DATA / "points.csv"
tri_out = OUT / "genova_osm_triangles.npy"
dem_out = OUT / "genova_area_dem.tif"
BUFFER_DEG = 0.02

lats, lons = [], []
with open(coords, newline="", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        lats.append(float(row.get("lat") or row.get("lat_deg")))
        lons.append(float(row.get("lon") or row.get("lon_deg")))

south = min(lats) - BUFFER_DEG
north = max(lats) + BUFFER_DEG
west = min(lons) - BUFFER_DEG
east = max(lons) + BUFFER_DEG
print(f"OSM bbox: S={south:.5f} W={west:.5f} N={north:.5f} E={east:.5f}")

from gnss_gpu.io.dem_download import download_dem_for_bbox
dem_path, ntiles = download_dem_for_bbox(
    south=south - 0.15, west=west - 0.15,
    north=north + 0.15, east=east + 0.15,
    output_tif=dem_out, zoom=12,
)
print("DEM:", dem_path, "tiles=", ntiles)

!python {REPO}/experiments/fetch_osm_buildings_bbox.py \
  --south {south} --west {west} --north {north} --east {east} \
  --cache-json {OUT}/genova_osm_cache.json \
  --out-triangles {tri_out} \
  --dem-path {dem_out} \
  --geoid-correction egm96

print("Mesh:", tri_out, "exists=", tri_out.exists())


## 6) Label LOS/NLOS

`is_los` = edifici ∧ elev mask ∧ non terrain-blocked  
`is_los_buildings` = solo edifici (ablation)

UTC allineato al BRDC (default Genua n5: `2026-10-09T12:00:00Z`).


In [ ]:
# 6) Run labeling (DEM + egm96)
import os, subprocess
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
os.environ.pop("PROJ_DATA", None)
os.environ.pop("PROJ_LIB", None)

COORDS = DATA / "points.csv"
NAV = next(DATA.glob("*MN.rnx"), None) or next(DATA.glob("*.rnx"), None)
TRIANGLES = OUT / "genova_osm_triangles.npy"
if not TRIANGLES.exists():
    TRIANGLES = next(DATA.glob("*triangles*.npy"), None)
DEM = OUT / "genova_area_dem.tif"

UTC = "2026-10-09T12:00:00Z"
ELEV_MASK = 10.0

assert COORDS.exists(), COORDS
assert NAV and Path(NAV).exists(), f"Manca NAV in {DATA}"
assert TRIANGLES and Path(TRIANGLES).exists(), "Manca mesh — lancia cella 5d"

cmd = [
    "python", "experiments/label_los_nlos_coords.py",
    "--coords", str(COORDS),
    "--nav", str(NAV),
    "--triangles", str(TRIANGLES),
    "--utc", UTC,
    "--rx-alt-mode", "dem",
    "--rx-ant-height-m", "1.5",
    "--geoid-correction", "egm96",
    "--elevation-mask-deg", str(ELEV_MASK),
    "--output-csv", str(OUT / "los_nlos_labels.csv"),
    "--summary-csv", str(OUT / "los_nlos_summary.csv"),
]
if DEM.exists():
    cmd += ["--dem-path", str(DEM)]
else:
    cmd += ["--dem-auto-download", "--dem-auto-out", str(OUT / "genova_area_dem.tif")]

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


In [ ]:
# 7) Anteprima labels
import pandas as pd
from pathlib import Path

OUT = Path("/content/los_work/results")
labels = pd.read_csv(OUT / "los_nlos_labels.csv")
summary = pd.read_csv(OUT / "los_nlos_summary.csv")
display(summary)
display(labels.head(20))
print(labels.groupby(["point_id", "is_los"]).size())
if "terrain_blocked_visible" in labels.columns:
    print("terrain_blocked_visible per punto:")
    print(labels.groupby("point_id")["terrain_blocked_visible"].sum())


## 8) Viewer Cesium (raggi LOS/NLOS)

Come i test HK: `build_stadium_visibility_viz.py` → HTML.

- Default viewer: **World Terrain** (non “Bare ellipsoid” a Genova)
- Nascondi OSM Ion
- Token: [Cesium ion](https://ion.cesium.com/)


In [ ]:
# 8a) Token Cesium ion
import os
os.environ["CESIUM_ION_TOKEN"] = "PASTE_YOUR_CESIUM_ION_TOKEN_HERE"
tok = os.environ.get("CESIUM_ION_TOKEN", "").strip()
print("CESIUM_ION_TOKEN set:", bool(tok) and "PASTE" not in tok)


In [ ]:
# 8b) Genera HTML Cesium (dt-s=1)
import os, csv, subprocess
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
os.environ["MPLBACKEND"] = "Agg"

NAV = next(DATA.glob("*MN.rnx"))
TRIANGLES = OUT / "genova_osm_triangles.npy"
SUMMARY = OUT / "los_nlos_summary.csv"
assert SUMMARY.exists(), "Lancia prima il labeling"

rows = list(csv.DictReader(open(SUMMARY, encoding="utf-8")))
viz_csv = OUT / "viz_points_dem_alt.csv"
with open(viz_csv, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["id", "lat", "lon", "alt"])
    w.writeheader()
    for r in rows:
        w.writerow({"id": r["point_id"], "lat": r["lat_deg"], "lon": r["lon_deg"], "alt": r["alt_m"]})

OUT_HTML = OUT / "genova_los_rays_viz.html"
UTC = "2026-10-09T12:00:00Z"
token = os.environ.get("CESIUM_ION_TOKEN", "").strip()

cmd = [
    "python", "experiments/build_stadium_visibility_viz.py",
    "--latlon-csv", str(viz_csv),
    "--triangles-npy", str(TRIANGLES),
    "--nav", str(NAV),
    "--out-html", str(OUT_HTML),
    "--reference-out", str(OUT / "genova_los_rays_viz_reference.csv"),
    "--area-name", "GenovaCoords",
    "--start-utc", UTC,
    "--dt-s", "1",
    "--n-epochs", str(len(rows)),
    "--traj-step", "1",
    "--elevation-mask-deg", "10",
    "--export-mesh-glb",
    "--plateau-glb-radius-m", "2500",
    "--plateau-glb-max-tris", "800000",
]
if token and "PASTE" not in token:
    cmd += ["--cesium-ion-token", token]

print(" ".join(cmd))
r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:] if r.stdout else "(no stdout)")
print(r.stderr[-1500:] if r.stderr else "(no stderr)")
r.check_returncode()
print("HTML:", OUT_HTML.exists())


In [ ]:
# 8c) Apri viewer (IFrame)
from pathlib import Path
from IPython.display import IFrame, display
import subprocess, time

OUT = Path("/content/los_work/results")
html = OUT / "genova_los_rays_viz.html"
assert html.exists(), html

subprocess.Popen(f"python -m http.server 8765 --directory {OUT}", shell=True)
time.sleep(1)
display(IFrame(src=f"http://localhost:8765/{html.name}", width="100%", height=700))
print("Se vuoto, apri:", f"http://localhost:8765/{html.name}")


## 9) Skyline + obstruction skyplot

`plot_rx_skyline.py`: mask azimut→elev (BVH edifici ± DEM) + tracce NAV.

Se i picchi edifici spariscono (solo DEM basso): mesh e RX non condividono `egm96` → rilancia **5d** poi questa cella.


In [ ]:
# 9) Skyline / skyplot / mask export
import os, subprocess
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
os.environ["MPLBACKEND"] = "Agg"

NAV = next(DATA.glob("*MN.rnx"))
cmd = [
    "python", "experiments/plot_rx_skyline.py",
    "--coords", str(DATA / "points.csv"),
    "--triangles", str(OUT / "genova_osm_triangles.npy"),
    "--rx-alt-mode", "dem",
    "--dem-path", str(OUT / "genova_area_dem.tif"),
    "--geoid-correction", "egm96",
    "--source", "both",
    "--nav", str(NAV),
    "--utc", "2026-10-09T12:00:00Z",
    "--duration-s", "7200",
    "--dt-s", "60",
    "--az-step-deg", "1",
    "--out-png", str(OUT / "rx_skyline.png"),
    "--out-skyplot-png", str(OUT / "rx_obstruction_skyplot.png"),
    "--out-mask-json", str(OUT / "rx_skyline_mask.json"),
    "--out-mask-pkl", str(OUT / "rx_skyline_mask.pkl"),
    "--out-csv", str(OUT / "rx_skyline_mask.csv"),
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


## 9b) Animazione presentazione (MP4)

Da RX → sweep mask → skyplot (+ tracce NAV). Dipendenze Colab sotto.

In [ ]:
# 9b) Install deps + render MP4 (un RX)
!pip install -q imageio imageio-ffmpeg contextily pillow

import os, subprocess
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
os.environ["MPLBACKEND"] = "Agg"

# pull se serve lo script nuovo
subprocess.run([
    "git", "-C", REPO, "pull",
    "https://github.com/andrea444460/gnss_gpu_los_nlos.git",
    "feature/cuda-preprocess-area-map",
], check=False)

NAV = next(DATA.glob("*MN.rnx"))
POINT_INDEX = 0  # cambia RX: 0..n-1
cmd = [
    "python", "experiments/animate_rx_skyline.py",
    "--mask-json", str(OUT / "rx_skyline_mask.json"),
    "--point-index", str(POINT_INDEX),
    "--dem-path", str(OUT / "genova_area_dem.tif"),
    "--geoid-correction", "egm96",
    "--nav", str(NAV),
    "--utc", "2026-10-09T12:00:00Z",
    "--duration-s", "7200",
    "--dt-s", "120",
    "--basemap",
    "--fov-zero-deg", "45",
    "--fps", "30",
    "--out-mp4", str(OUT / f"rx_mask_anim_p{POINT_INDEX}.mp4"),
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)
print("MP4:", OUT / f"rx_mask_anim_p{POINT_INDEX}.mp4")

## 10) Download risultati


In [ ]:
# 10) Download
from google.colab import files
from pathlib import Path

OUT = Path("/content/los_work/results")
for name in (
    "los_nlos_labels.csv",
    "los_nlos_summary.csv",
    "genova_los_rays_viz.html",
    "genova_los_rays_viz_mesh.glb",
    "rx_skyline.png",
    "rx_obstruction_skyplot.png",
    "rx_skyline_mask.json",
    "rx_skyline_mask.pkl",
    "rx_skyline_mask.csv",
    "rx_mask_anim_p0.mp4",
):
    p = OUT / name
    if p.exists():
        files.download(str(p))
        print("↓", p.name)
    else:
        print("skip:", name)
